### Prompt Chaining
- Decomposes complex task into a sequence of steps, each step is executed by an agent. 
- One agent's output will be the input to the next one in sequence.

In [3]:
import os
from dotenv import load_dotenv
from langgraph.graph import StateGraph, START, END
from typing_extensions import TypedDict
from langchain_openai import ChatOpenAI

load_dotenv()

True

In [4]:
model = ChatOpenAI(model="gpt-4o-mini", temperature=0.5)

In [5]:
from typing import Optional
from pydantic import BaseModel, Field


class Specification(BaseModel):
    cpu: Optional[str] = Field(description="The CPU specification of the computer.")
    memory: Optional[str] = Field(description="The memory specification of the computer.")
    storage: Optional[str] = Field(description="The storage specification of the computer.")

class AgentState(TypedDict):
    input_text: str
    spec_text: str
    formatted_spec: Specification

In [10]:
# Node 1: Extract Specifications
def extract_specifications(state: AgentState) -> AgentState:
    prompt = f"""
    Extract the technical specifications from following text:

    {state['input_text']}
    """
    response = model.invoke(prompt)
    state['spec_text'] = response
    print(f"Extraction - extracted Specifications: { state['spec_text']}")
    return state

In [11]:
# Node 2: Format Specifications
def format_specifications(state: AgentState) -> AgentState:
    prompt = f"""
    Generate formatted specification from following text:

    {state['spec_text']}
    """
    formatted_spec = model.with_structured_output(Specification).invoke(prompt)
    state['formatted_spec'] = formatted_spec
    print(f"Formatting - formatted Specifications: { state['formatted_spec']}")
    return state

In [12]:
workflow = StateGraph(AgentState)
workflow.add_node("extract", extract_specifications)
workflow.add_node("format", format_specifications)

workflow.set_entry_point("extract")
workflow.add_edge("extract", "format")
workflow.add_edge("format", END)

graph = workflow.compile()

In [15]:
input_text = "The new laptop model features a 3.5 GHz octa-core processor, 16GB of RAM, and a 1TB NVMe SSD."

initial_state: AgentState = {
    "input_text": input_text
}

result_state = graph.invoke(initial_state)

print("Final Result:", result_state)
print("Formatted Specifications:", result_state['formatted_spec'])
print(type(result_state['formatted_spec']))

Extraction - extracted Specifications: content='**Technical Specifications:**\n\n- Processor: 3.5 GHz octa-core\n- RAM: 16GB\n- Storage: 1TB NVMe SSD' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 33, 'prompt_tokens': 50, 'total_tokens': 83, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_1590f93f9d', 'id': 'chatcmpl-D4ExnhStl3IHmpiphFgkFNHL4tK14', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019c1673-a7dc-70a1-b802-5fefac97dd49-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 50, 'output_tokens': 33, 'total_tokens': 83, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}
Formatti